<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Notebook_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook Generator — Scaffold New 7-Step AEI Notebooks

A meta-tool for adding new models to the AEI-Colab-Notebooks suite. Fill in the spec form in Step 4, click **Generate**, and a fully-formed 7-step Colab notebook (with Drive cache, lazy-loaded engine wrappers, info= tooltips, demo.load welcome, batch try/except, and the same `step1-install` .. `step7-batch` pattern used by every other notebook in the repo) gets written to `/content/<Name>_Colab.ipynb`.

Use it to bootstrap a new TTS / 3D / image / video model notebook in under 2 minutes. After generation, open the new notebook, fill in the model-specific `synth_*` / `gen_*` functions in Step 3, and the UI works out of the box.

**What gets generated** (9 cells, all following the project standard):

1. Colab-open badge
2. Markdown header (title, subtitle, parameters, license, quick start, optional **More info** block with HF/GitHub/arXiv/citation)
3. `STEP 1 — Install Dependencies` (form-collapsed; HF cache prologue, version-pinned install)
4. `STEP 2 — Pre-cache Models` (form-collapsed; Drive-cached, progress logged)
5. `STEP 3 — Core Functions` (form-collapsed; lazy-load engine wrapper, `synth` / `gen` functions)
6. `STEP 4 — Gradio UI` (form-collapsed; tabs for TTS/3D variants, `info=` tooltips, `concurrency_limit=2`, `clear_output()`, `demo.load` welcome)
7. `STEP 5 — Keep-Alive` (form-collapsed; 60-s pinger thread)
8. `STEP 6 — Quick Test` (form-collapsed; one-off inference with FileLink display)
9. `STEP 7 — Batch` (form-collapsed; per-iteration try/except, per-item output file)

The output follows the same 9-cell shape as the other 16 notebooks in the repo, so it passes the same audit and integrates cleanly with `TTS_Model_Loader.ipynb` / `TTS_Voice_Library.ipynb`.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown The generator itself only needs Gradio (and a few stdlib helpers).
# @markdown Subsequent runs are instant (idempotent).

import os, sys, json, subprocess

if 'gradio' not in sys.modules:
    print('[pip] Installing Gradio (only the generator needs it) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==5.49.1', 'huggingface-hub>=0.25'], check=True)
else:
    print('[pip] Gradio already installed, skipping.')

import gradio as gr
print('[Setup] Generator ready.')

In [ ]:

# @title STEP 2 — Generator Engine
# @markdown The Python code that turns a spec dict into a 9-cell notebook JSON.
# @markdown Skip reading this cell — just run it and move to Step 4 to use the form.

import os, json, textwrap, datetime, re
from pathlib import Path

# ---- Spec validation helpers -----------------------------------
MODALITIES = ['tts', '3d', 'image', 'video']
LICENSES = ['Apache 2.0', 'MIT', 'OpenRAIL-M', 'OpenRAIL++', 'Research / Non-Commercial', 'CC-BY-NC', 'CC-BY-4.0', 'Other']
SAMPLE_RATES = [8000, 16000, 22050, 24000, 32000, 44100, 48000]
HF = chr(34)  # avoid quote-escaping headaches below


def _slug(name):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', name.strip()).strip('-')
    return s or 'Model'


def _validate_spec(spec):
    errs = []
    if not spec.get('name'):
        errs.append('name is required')
    if spec.get('modality') not in MODALITIES:
        errs.append('modality must be one of ' + str(MODALITIES))
    if not spec.get('hf_repo'):
        errs.append('hf_repo is required (e.g. tencent/Hunyuan3D-2)')
    if spec.get('modality') == 'tts' and spec.get('sample_rate') not in SAMPLE_RATES:
        errs.append('sample_rate must be one of ' + str(SAMPLE_RATES))
    if spec.get('license') not in LICENSES:
        errs.append('license must be one of ' + str(LICENSES))
    if not spec.get('subtitle'):
        errs.append('subtitle is required')
    return errs


def _add_nl(lines):
    if not lines:
        return lines
    return [s + '\n' for s in lines[:-1]] + [lines[-1]]


def _badge(fname):
    return ['<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/' + fname + '" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>']


def _header_md(spec):
    name = spec['name']
    title = spec.get('title') or name
    subtitle = spec['subtitle']
    params = spec.get('params', '?')
    weight = spec.get('weight_size', '?')
    langs = spec.get('languages', '?')
    lic = spec.get('license', '?')
    extra = spec.get('extra_features', '')
    repo = spec['hf_repo']
    sub = spec.get('subfolder', '')
    arxiv = spec.get('arxiv_url', '').strip()
    github = spec.get('github_url', '').strip()
    hf_card = spec.get('hf_card_url', '').strip() or ('https://huggingface.co/' + repo)
    cite = spec.get('citation', '').strip()

    D = HF
    repo_disp = repo + (' / `' + sub + '`' if sub else '')

    lines = ['## ' + title, '', subtitle, '']
    bullet_lines = [
        '- **' + name + '** (`' + repo_disp + '`) — ' + params + ' parameters, ~' + weight + ' on disk.',
    ]
    if spec['modality'] == 'tts':
        bullet_lines.append('- **Sample rate:** ' + str(spec['sample_rate']) + ' Hz, **' + langs + '** language(s).')
    else:
        bullet_lines.append('- **' + spec['modality'].upper() + ' modality.**')
    bullet_lines.append('- **License:** ' + lic + '.')
    if extra:
        bullet_lines.append('')
        bullet_lines.append(extra)
    bullet_lines.append('')
    bullet_lines.append('All weights are pulled from Hugging Face and cached on your Google Drive so subsequent runs are instant. The notebook follows the standard 7-step pattern: **Install → Pre-cache → Core functions → Gradio UI → Keep-alive → Quick test → Batch**.')
    bullet_lines.append('')
    bullet_lines.append('**Disclaimer:** Generated outputs are not moderated by any safety system. You are responsible for downstream use under the model license.')
    bullet_lines.append('')
    bullet_lines.append('### Quick Start')
    bullet_lines.append('')
    bullet_lines.append('1. Connect to a GPU runtime (A100/L4 recommended; T4 may OOM for larger models).')
    bullet_lines.append('2. Run Steps 1-4 in order — Step 1 installs, Step 2 caches weights, Step 3 defines the engine, Step 4 launches the UI.')
    bullet_lines.append('3. Open the Gradio link from Step 4 and start generating.')
    # More info section
    if arxiv or github or hf_card or cite:
        bullet_lines.append('')
        bullet_lines.append('### More info')
        bullet_lines.append('')
        if hf_card:
            bullet_lines.append('- [Hugging Face model card](' + hf_card + ')')
        if github:
            bullet_lines.append('- [GitHub repository](' + github + ')')
        if arxiv:
            bullet_lines.append('- [arXiv paper](' + arxiv + ')')
        if cite:
            bullet_lines.append('')
            bullet_lines.append('**Citation:**')
            bullet_lines.append('')
            bullet_lines.append('```bibtex')
            bullet_lines.append(cite)
            bullet_lines.append('```')
    return lines + bullet_lines


def _step1_install(spec):
    name = spec['name']
    deps = spec.get('pip_deps', '').strip() or 'transformers accelerate'
    gated = spec.get('gated', False)
    requires_gpu = spec.get('requires_gpu', True)
    lines = [
        '# @title STEP 1 — Install Dependencies',
        '# @markdown Installs the `' + name + '` Python package and its dependencies. First run: ~2-5 min; subsequent runs: instant.',
        '',
        'import os, sys, subprocess, time',
        '',
    ]
    if requires_gpu:
        lines += [
            'if not torch.cuda.is_available():',
            "    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime \\u2192 Change runtime type \\u2192 L4 or A100')",
            '',
            'gpu_name = torch.cuda.get_device_name(0)',
            'vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3',
            "print(f'[GPU] {gpu_name} \\u2014 {vram_gb:.0f} GB')",
            '',
        ]
    else:
        lines += [
            "print('[Runtime] CPU-only model. No GPU required.')",
            '',
        ]
    if gated:
        lines += [
            '# Read HF_TOKEN from Colab secrets for gated models.',
            'try:',
            '    from google.colab import userdata',
            "    HF_TOKEN = userdata.get('HF_TOKEN')",
            "    if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN; print('[HF] HF_TOKEN loaded.')",
            "    else: print('[HF] HF_TOKEN not set \\u2014 gated downloads will fail.')",
            'except Exception as e:',
            "    print(f'[HF] Could not read secrets: {e}')",
            '',
        ]
    lines += [
        't0 = time.time()',
    ]
    for dep in [d.strip() for d in deps.split(chr(10)) if d.strip()]:
        lines.append("subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '" + dep + "'], check=True)")
    lines += [
        "print(f'[Install] DONE in {time.time() - t0:.1f}s.')",
        '',
        '# Sanity-check the install.',
        "print('[Install] OK')",
    ]
    return lines


def _step2_cache(spec):
    name = spec['name']
    repo = spec['hf_repo']
    sub = spec.get('subfolder', '').strip()
    weight = spec.get('weight_size', '?')
    lines = [
        '# @title STEP 2 — Pre-cache Models',
        '# @markdown Downloads the `' + name + '` weights to Google Drive so subsequent runs are instant.',
        '',
        'import os, sys, pathlib, time',
        'from huggingface_hub import snapshot_download, hf_hub_download',
        'try:',
        '    from tqdm.notebook import tqdm',
        'except ImportError:',
        '    tqdm = None',
        '',
        "CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')",
        'CACHE_ROOT.mkdir(parents=True, exist_ok=True)',
        "os.environ['HF_HOME'] = str(CACHE_ROOT.parent)",
        "os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)",
        "print(f'[Cache] HF cache root: {CACHE_ROOT}')",
        '',
        "REPO = '" + repo + "'",
    ]
    if sub:
        lines.append("SUBFOLDER = '" + sub + "'")
    else:
        lines.append('SUBFOLDER = None')
    lines += [
        '',
        'def _human(n):',
        "    for u in ['B', 'KB', 'MB', 'GB', 'TB']:",
        "        if n < 1024: return f'{n:.1f} {u}'",
        '        n /= 1024',
        "    return f'{n:.1f} PB'",
        '',
        "print(f'[Download] {REPO} (" + weight + ") ...')",
        't0 = time.time()',
        'try:',
        '    if SUBFOLDER:',
        '        snapshot_download(',
        "            repo_id=REPO, allow_patterns=[f'{SUBFOLDER}/*'] + ['*.json', '*.txt'],",
        '            tqdm_class=tqdm,',
        '        )',
        '    else:',
        '        snapshot_download(repo_id=REPO, tqdm_class=tqdm)',
        "    print(f'[Download] Done in {time.time() - t0:.1f}s.')",
        'except Exception as e:',
        "    print(f'[Download] FAILED: {e}')",
        '    raise',
    ]
    return lines


def _step3_core(spec):
    name = spec['name']
    sub = spec.get('subfolder', '').strip()
    sample_rate = spec.get('sample_rate', 24000)
    modality = spec['modality']
    cls = name.replace('-', '').replace('.', '').replace(' ', '') + 'Engine'
    lines = [
        '# @title STEP 3 — Core Functions (Lazy Model Loading)',
        '# @markdown Defines a `' + cls + '` class that holds the `' + name + '` pipeline.',
        "# @markdown Edit the `load` and `infer` methods to match the model's actual API.",
        '',
        'import os, sys, time, gc, json, traceback',
        'import torch, numpy as np',
        'from pathlib import Path',
    ]
    if modality == 'tts':
        lines.append('SAMPLE_RATE = ' + str(sample_rate))
    elif modality == '3d':
        lines.append('import trimesh')
    lines += [
        '',
        "REPO = '" + spec['hf_repo'] + "'",
    ]
    if sub:
        lines.append("SUBFOLDER = '" + sub + "'")
    else:
        lines.append('SUBFOLDER = None')
    lines += [
        '',
        'class ' + cls + ':',
        '    """Lazy loader for the model. Edit the `load` method to use the actual model API."""',
        '    def __init__(self):',
        '        self.pipeline = None',
        '',
        '    def load(self):',
        '        if self.pipeline is not None: return',
        '        if torch.cuda.is_available(): torch.cuda.empty_cache()',
        "        print(f'[Load] ' + chr(123) + 'REPO' + chr(125) + ' ...')",
        '        t0 = time.time()',
        '        try:',
        '            # === TODO: replace this with the actual model loader ===',
        "            from transformers import AutoModel  # placeholder",
        '            self.pipeline = AutoModel.from_pretrained(',
        "                REPO, subfolder=SUBFOLDER, device_map='cuda', torch_dtype=torch.bfloat16,",
        '            )',
        '            # ====================================================',
        '        except Exception as e:',
        '            print(f"[Load] FAILED: {e}"); raise',
        '        print(f"[Load] Ready in {time.time() - t0:.1f}s. VRAM: " + chr(123) + "torch.cuda.memory_allocated()/1024**3:.1f" + chr(125) + " GB")',
        '',
        '    def unload(self):',
        '        if self.pipeline is not None:',
        '            del self.pipeline; self.pipeline = None',
        '            gc.collect(); torch.cuda.empty_cache()',
        '',
        '    def infer(self, *args, **kwargs):',
        '        """Run inference. Returns whatever the UI needs (audio path, mesh path, etc)."""',
        '        self.load()',
        '        # === TODO: implement the actual inference call ===',
        "        raise NotImplementedError('Fill in the model\\'s actual inference call here.')",
        '',
        'ENGINE = ' + cls + '()',
        '',
        "print('[Core] Ready. Engine loads on first use.')",
    ]
    return lines


def _step4_ui(spec):
    name = spec['name']
    sample_rate = spec.get('sample_rate', 24000)
    modality = spec['modality']
    quick_lines_raw = spec.get('quick_examples', '').strip()
    quick_lines = [l for l in quick_lines_raw.split(chr(10)) if l.strip()][:5]
    quick_repr = '[' + ', '.join(repr(l) for l in quick_lines) + ']'
    repo = spec['hf_repo']
    params = spec.get('params', '?')
    subtitle = spec['subtitle']

    status_msg = 'Welcome to ' + name + '. Edit Step 3 to wire up ENGINE.infer, then come back here.'

    lines = [
        '# @title STEP 4 — Launch Gradio UI',
        '# @markdown Opens a Gradio app for the `' + name + '` model.',
        '',
        'import os, sys, time, json, uuid, shutil, pathlib, gc, traceback, random, tempfile',
        'import torch, numpy as np, gradio as gr',
        'from IPython.display import clear_output, display, FileLink',
        '',
        "OUT_DIR = pathlib.Path(tempfile.gettempdir()) / 'aei_" + name.lower().replace(' ', '_') + "_out'",
        'OUT_DIR.mkdir(parents=True, exist_ok=True)',
        '',
    ]

    if modality == 'tts':
        lines += [
            'SAMPLE_RATE = ' + str(sample_rate),
            '',
            'def _new_out():',
            "    p = OUT_DIR / f'{int(time.time())}_{uuid.uuid4().hex[:6]}.wav'",
            '    return p',
            '',
            'def do_synthesize(text, progress=gr.Progress()):',
            "    if not text or not text.strip(): return None, '\\u274c Text is empty.'",
            '    try:',
            "        progress(0.1, desc='Loading model ...')",
            '        ENGINE.load()',
            "        progress(0.3, desc='Synthesizing ...')",
            '        t0 = time.time()',
            '        # === TODO: wire up to ENGINE.infer ===',
            "        # out_path = ENGINE.infer(text.strip())",
            "        raise NotImplementedError('Wire up to ENGINE.infer in Step 3 first.')",
            "        progress(1.0, desc='Done.')",
            "        return out_path, f'\\u2705 {time.time() - t0:.1f}s @ {SAMPLE_RATE} Hz'",
            '    except Exception as e:',
            '        traceback.print_exc()',
            "        return None, f'\\u274c {e}'",
            '',
            "with gr.Blocks(title='" + name + " Colab', theme=gr.themes.Soft()) as demo:",
            '    gr.Markdown(f"# ' + name + ' Colab\\n**Model:** `' + repo + '` (' + params + ')\\n\\n' + subtitle + '")',
            '    with gr.Row():',
            '        with gr.Column():',
            '            text_in = gr.Textbox(',
            "                label='Text to synthesize', value='Hello! This is a test of the synthesis pipeline.',",
            '                lines=4,',
            "                info='Up to a few hundred characters works well; longer is fine but slower.'",
            '            )',
            "            btn = gr.Button('Generate', variant='primary')",
            '        with gr.Column():',
            "            audio_out = gr.Audio(label='Output', type='filepath')",
            "            status = gr.Markdown('')",
            '    btn.click(do_synthesize, inputs=[text_in], outputs=[audio_out, status])',
        ]
    elif modality == '3d':
        lines += [
            'def do_generate(image, progress=gr.Progress()):',
            "    if image is None: return None, '\\u274c Upload an image first.'",
            '    try:',
            "        progress(0.1, desc='Loading model ...')",
            '        ENGINE.load()',
            "        progress(0.4, desc='Generating shape ...')",
            '        t0 = time.time()',
            "        out_path = OUT_DIR / f'{int(time.time())}_{uuid.uuid4().hex[:6]}.glb'",
            '        # === TODO: wire up to ENGINE.infer ===',
            '        # out_path = ENGINE.infer(image)',
            "        raise NotImplementedError('Wire up to ENGINE.infer in Step 3 first.')",
            "        progress(1.0, desc='Done.')",
            "        return str(out_path), f'\\u2705 {time.time() - t0:.1f}s'",
            '    except Exception as e:',
            '        traceback.print_exc()',
            "        return None, f'\\u274c {e}'",
            '',
            "with gr.Blocks(title='" + name + " Colab', theme=gr.themes.Soft()) as demo:",
            '    gr.Markdown(f"# ' + name + ' Colab\\n**Model:** `' + repo + '` (' + params + ')\\n\\n' + subtitle + '")',
            '    with gr.Row():',
            '        with gr.Column():',
            "            img_in = gr.Image(label='Input image', type='pil', image_mode='RGBA', height=320,",
            ")",
            "            btn = gr.Button('Generate', variant='primary')",
            '        with gr.Column():',
            "            file_out = gr.File(label='Output .glb')",
            "            status = gr.Markdown('')",
            '    btn.click(do_generate, inputs=[img_in], outputs=[file_out, status])',
        ]
    else:
        lines += [
            'def do_run(*inputs, progress=gr.Progress()):',
            '    try:',
            "        progress(0.2, desc='Loading ...')",
            '        ENGINE.load()',
            "        progress(0.5, desc='Running ...')",
            '        t0 = time.time()',
            '        # === TODO: wire up to ENGINE.infer ===',
            "        raise NotImplementedError('Wire up to ENGINE.infer in Step 3 first.')",
            "        progress(1.0, desc='Done.')",
            "        return out_path, f'\\u2705 {time.time() - t0:.1f}s'",
            '    except Exception as e:',
            '        traceback.print_exc()',
            "        return None, f'\\u274c {e}'",
            '',
            "with gr.Blocks(title='" + name + " Colab', theme=gr.themes.Soft()) as demo:",
            '    gr.Markdown(f"# ' + name + ' Colab\\n**Model:** `' + repo + '` (' + params + ')\\n\\n' + subtitle + '")',
            '    with gr.Row():',
            '        with gr.Column():',
            "            btn = gr.Button('Run', variant='primary')",
            "            status = gr.Markdown('')",
            '    btn.click(do_run, outputs=[status])',
        ]

    lines += [
        '',
        "    with gr.Tab('VRAM'):",
        "        gr.Markdown('**Free the model from VRAM** to make room for another engine.')",
        "        vram_text = gr.Textbox(label='Status', value=lambda: f'Allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB' if torch.cuda.is_available() else 'No GPU', interactive=False, lines=3)",
        '        with gr.Row():',
        "            gr.Button('Free engine').click(lambda: (ENGINE.unload(), f'Freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.')[1], outputs=vram_text)",
        "            gr.Button('Refresh status').click(lambda: f'Allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB' if torch.cuda.is_available() else 'No GPU', outputs=vram_text)",
        '',
        'if ' + repr(quick_lines) + ':',
        '    gr.Examples(',
        '        examples=[[l] for l in ' + repr(quick_lines) + '],',
        "        inputs=[text_in] if modality == 'tts' else [img_in],",
        "        label='Quick starts', examples_per_page=5,",
        '    )',
        '',
        'clear_output()',
        "print('[UI] Launching ...')",
        'demo.queue(max_size=4, default_concurrency_limit=2)',
        'demo.load(lambda: "' + status_msg.replace('"', '\\"') + '", outputs=[status])',
        'demo.launch(share=True, show_error=True, height=720, prevent_thread_lock=False)',
    ]
    return lines


def _step5_keepalive():
    return [
        '# @title STEP 5 — Keep-Alive (anti-disconnect)',
        '# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.',
        '',
        'import time, requests, datetime, threading, IPython',
        '_stop = threading.Event()',
        'def _pinger():',
        '    while not _stop.is_set():',
        '        try:',
        "            requests.get('https://huggingface.co/api/models/tencent/Hunyuan3D-2', timeout=5)",
        '            IPython.display.clear_output(wait=True)',
        "            print(f'[Keep-Alive] {datetime.datetime.now().strftime(\"%H:%M:%S\")} OK. Stop with `_stop.set()`.')",
        '        except Exception as e:',
        "            print(f'[Keep-Alive] {datetime.datetime.now().strftime(\"%H:%M:%S\")} WARN: {e}')",
        '        _stop.wait(60)',
        '_t = threading.Thread(target=_pinger, daemon=True)',
        '_t.start()',
        "print('[Keep-Alive] Started.')",
    ]


def _step6_quicktest(spec):
    name = spec['name']
    sample_rate = spec.get('sample_rate', 24000)
    modality = spec['modality']
    quick_lines = [l for l in spec.get('quick_examples', '').strip().split(chr(10)) if l.strip()][:1]
    default_text = quick_lines[0] if quick_lines else ('Hello world.' if modality == 'tts' else 'A small red cube.')

    if modality == 'tts':
        return [
            '# @title STEP 6 — Quick Test (one-off inference, no UI)',
            '# @markdown Run a single ' + name + ' inference from the notebook.',
            '',
            "TEXT = " + repr(default_text) + "  # @param {type:\"string\"}",
            '',
            'ENGINE.load()',
            't0 = time.time()',
            '# === TODO: wire up to ENGINE.infer ===',
            "out_path = ENGINE.infer(TEXT)",
            "print(f'[Done] {out_path} ({time.time() - t0:.1f}s, " + str(sample_rate) + " Hz)')",
            "from IPython.display import Audio, FileLink, display",
            "display(Audio(out_path, rate=" + str(sample_rate) + "))",
            "display(FileLink(out_path))",
        ]
    else:
        return [
            '# @title STEP 6 — Quick Test (one-off inference, no UI)',
            '# @markdown Run a single ' + name + ' inference from the notebook.',
            '',
            "PROMPT = " + repr(default_text) + "  # @param {type:\"string\"}",
            '',
            'ENGINE.load()',
            't0 = time.time()',
            '# === TODO: wire up to ENGINE.infer ===',
            "out_path = ENGINE.infer(PROMPT)",
            "print(f'[Done] {out_path} ({time.time() - t0:.1f}s)')",
            "from IPython.display import FileLink, display",
            "display(FileLink(out_path))",
        ]


def _step7_batch(spec):
    name = spec['name']
    sample_rate = spec.get('sample_rate', 24000)
    modality = spec['modality']
    quick_lines = [l for l in spec.get('quick_examples', '').strip().split(chr(10)) if l.strip()][:5]
    default_batch = chr(10).join(quick_lines) if quick_lines else 'Line 1' + chr(10) + 'Line 2' + chr(10) + 'Line 3'
    out_ext = '.wav' if modality == 'tts' else '.glb'
    return [
        '# @title STEP 7 — Batch Generation',
        '# @markdown Read a list of inputs (one per line) and generate outputs for each.',
        '',
        'import os, time, pathlib, traceback',
        'BATCH_INPUTS = """' + chr(92) + chr(10) + default_batch.replace('"""', '\\"\\"\\"') + chr(10) + '"""',
        'lines = [l.strip() for l in BATCH_INPUTS.splitlines() if l.strip()]',
        "print(f'[Batch] {len(lines)} item(s).')",
        "out_dir = pathlib.Path('/content/aei_" + name.lower().replace(' ', '_') + "_batch')",
        'out_dir.mkdir(parents=True, exist_ok=True)',
        'ok = 0; fail = 0',
        'ENGINE.load()',
        'for i, line in enumerate(lines, 1):',
        '    try:',
        '        t0 = time.time()',
        '        # === TODO: wire up to ENGINE.infer ===',
        "        out_path = ENGINE.infer(line)",
        "        safe = ''.join(c if c.isalnum() else '_' for c in line[:30]).strip('_') or f'item_{i:02d}'",
        "        final = out_dir / f'{i:03d}_{safe}" + out_ext + "'",
        '        # shutil.move(out_path, final)  # uncomment if your ENGINE writes to a temp path',
        "        print(f'  [{i:03d}] {line[:60]} -> {final.name} ({time.time() - t0:.1f}s)')",
        '        ok += 1',
        '        if torch.cuda.is_available(): torch.cuda.empty_cache()',
        '    except Exception as e:',
        "        print(f'  [{i:03d}] EXCEPTION: {e}'); traceback.print_exc(); fail += 1",
        "print(f'\\n[Batch] DONE: {ok} OK / {fail} failed / {len(lines)} total -> {out_dir}')",
    ]


# ---- Master assemble function ----------------------------------
def build_notebook(spec):
    errs = _validate_spec(spec)
    if errs:
        return None, errs
    name = spec['name']
    fname = _slug(name) + '_Colab.ipynb'

    cells = [
        {'cell_type': 'markdown', 'metadata': {'id': 'view-in-github', 'colab_type': 'text'},
         'source': _add_nl(_badge(fname))},
        {'cell_type': 'markdown', 'metadata': {'id': 'header'},
         'source': _add_nl(_header_md(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step1-install', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step1_install(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step2-cache', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step2_cache(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step3-core', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step3_core(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step4-ui', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step4_ui(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step5-keepalive', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step5_keepalive())},
        {'cell_type': 'code', 'metadata': {'id': 'step6-quicktest', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step6_quicktest(spec))},
        {'cell_type': 'code', 'metadata': {'id': 'step7-batch', 'cellView': 'form'},
         'execution_count': None, 'outputs': [],
         'source': _add_nl(_step7_batch(spec))},
    ]
    nb = {
        'cells': cells,
        'metadata': {
            'kernelspec': {'display_name': 'Python 3', 'name': 'python3', 'language': 'python'},
            'colab': {'provenance': [], 'collapsed_sections': []},
        },
        'nbformat': 4, 'nbformat_minor': 0,
    }
    return nb, []


def write_notebook(nb, out_dir='/content'):
    title = ''.join(nb['cells'][1]['source']).split(chr(10))[0].lstrip('# ').strip()
    short = re.split(r'\s+[\u2014:\u2013\-]\s+', title, 1)[0].strip()
    fname = _slug(short) + '_Colab.ipynb'
    out_path = Path(out_dir) / fname
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
    return str(out_path), fname


print('[Engine] Loaded. Use Step 4 to generate a notebook.')

In [ ]:

# @title STEP 3 — Preset Specs
# @markdown Curated model specs that you can use as starting points. Edit them in Step 4
# @markdown or load one and tweak.

PRESETS = {
    'Qwen3-TTS-1.7B': {
        'name': 'Qwen3-TTS-1.7B',
        'title': 'Qwen3-TTS-1.7B — Flagship Apache-2.0 Multilingual TTS',
        'subtitle': "Qwen's 1.7B TTS model. 10 languages, 9 premium speakers, voice-clone mode, voice-design mode. Apache 2.0.",
        'modality': 'tts',
        'hf_repo': 'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice',
        'subfolder': '',
        'params': '1.7B',
        'weight_size': '3.5 GB',
        'sample_rate': 24000,
        'languages': '10 (EN/ZH/JA/KO/DE/FR/RU/PT/ES/IT)',
        'license': 'Apache 2.0',
        'pip_deps': 'qwen-tts\ntransformers\naccelerate\nflash-attn --no-build-isolation',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Three modes**: 9 premium speakers, text-prompted voice design, and 3-second voice cloning. The flagship open TTS of the suite.',
        'quick_examples': 'Hello! Welcome to Qwen3-TTS.\nThe quick brown fox jumps over the lazy dog.\n哥哥，你回来啦，人家等了你好久好久了，要抱抱！\n今日は経済訓練の日です。',
        'hf_card_url': 'https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice',
        'github_url': 'https://github.com/QwenLM/Qwen3-TTS',
        'arxiv_url': '',
        'citation': '@misc{qwen3tts2025,\n  title={Qwen3-TTS: Apache-2.0 Multilingual Text-to-Speech},\n  author={Qwen Team},\n  year={2025}\n}',
    },
    'Higgs-Audio-v3': {
        'name': 'Higgs-Audio-v3',
        'title': 'Higgs-Audio-v3 — 4B Conversational TTS',
        'subtitle': '4 B conversational TTS from Boson AI. 100+ languages, voice cloning, and audio understanding in one model. Research/Non-Commercial.',
        'modality': 'tts',
        'hf_repo': 'multimodalart/higgs-audio-v3-tts-4b-transformers',
        'subfolder': '',
        'params': '4B',
        'weight_size': '10 GB',
        'sample_rate': 24000,
        'languages': '100+',
        'license': 'Research / Non-Commercial',
        'pip_deps': 'transformers\naccelerate\nflash-attn --no-build-isolation\noutetts>=0.4.0\nlangid',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Voice cloning** from a 3+ second reference clip. Multi-speaker dialogue support.',
        'quick_examples': "Hello! This is a test of Higgs audio v3.\nBonjour, je m'appelle Higgs.\n今天天氣真好，你吃了嗎？",
    },
    'MisoTTS-8B': {
        'name': 'MisoTTS-8B',
        'title': 'MisoTTS 8B — Sesame CSM Conversational TTS',
        'subtitle': '8 B Sesame CSM-1B model. Conversational turn-taking, natural prosody, English only. Other license (see model card).',
        'modality': 'tts',
        'hf_repo': 'MisoLabs/MisoTTS',
        'subfolder': '',
        'params': '8B',
        'weight_size': '~30 GB (all sub-models)',
        'sample_rate': 24000,
        'languages': 'English',
        'license': 'Other',
        'pip_deps': 'moshi==0.2.2\nsilentcipher @ git+https://github.com/SesameAILabs/silentcipher@d46d7d0893a583d8968ab3a6626e2289faec9152\nhuggingface-hub\nPillow',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Conversational** by design. Feed a sequence of `[text, audio]` turns and it continues the conversation naturally.',
        'quick_examples': 'I had a really great time at the conference last week.\nWhat do you think about the new policy?\nI am not sure if I should take the job or stay in school.',
    },
    'Trellis-3D': {
        'name': 'Trellis-3D',
        'title': "Trellis \u2014 Microsoft's Structured 3D Generation",
        'subtitle': 'Structured 3D generation (Gaussian splats + meshes + NeRFs from a single image) from Microsoft Research. ~12 GB, A100/L4 recommended.',
        'modality': '3d',
        'hf_repo': 'JeffreyXiang/TRELLIS-image-large',
        'subfolder': '',
        'params': '~1.2 B',
        'weight_size': '12 GB',
        'sample_rate': 0,
        'languages': '-',
        'license': 'MIT',
        'pip_deps': 'torch\ntorchvision\ntrimesh\ngradio\nhuggingface-hub\nimageio\npyvista',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Outputs**: textured Gaussian splats, extracted mesh, radiance field, or all three.',
        'quick_examples': 'A small red cube.\nA ceramic teapot.\nA cartoon-style dragon.',
    },
    'Step1X-3D': {
        'name': 'Step1X-3D',
        'title': 'Step1X-3D \u2014 High-Fidelity Image-to-3D',
        'subtitle': 'High-fidelity image-to-3D from StepFun. Multi-view diffusion + mesh refinement. ~10 GB, A100/L4 recommended.',
        'modality': '3d',
        'hf_repo': 'stepfun-ai/Step1X-3D',
        'subfolder': '',
        'params': '~4 B',
        'weight_size': '10 GB',
        'sample_rate': 0,
        'languages': '-',
        'license': 'Apache 2.0',
        'pip_deps': 'torch\ntrimesh\ngradio\nhuggingface-hub\npyvista\nkaolin',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**High-detail** mesh output. Per-instance texture baking.',
        'quick_examples': 'A 3D model of a cute cat, white background.\nA small ceramic teapot, soft lighting.\nA cartoon dragon character.',
    },
    'TripoSG': {
        'name': 'TripoSG',
        'title': 'TripoSG \u2014 VAST-AI-Research Image-to-3D',
        'subtitle': 'Image-to-3D from VAST-AI-Research. Fast DiT-based shape generation with high-quality mesh output. ~3 GB, fits T4.',
        'modality': '3d',
        'hf_repo': 'VAST-AI/TripoSG',
        'subfolder': '',
        'params': '~1.5 B',
        'weight_size': '3 GB',
        'sample_rate': 0,
        'languages': '-',
        'license': 'MIT',
        'pip_deps': 'torch\ntrimesh\ngradio\nhuggingface-hub\npyvista',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Single-GPU fast** generation.',
        'quick_examples': 'A small red cube.\nA wooden chair.\nA cartoon dragon.',
    },
    'Wan2.1-T2V': {
        'name': 'Wan2.1-T2V',
        'title': 'Wan 2.1 \u2014 Alibaba Text-to-Video',
        'subtitle': '1.3 B / 14 B text-to-video from Alibaba (Wan Team). Apache 2.0. Generates 5-10 second 720p clips from a text prompt.',
        'modality': 'video',
        'hf_repo': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers',
        'subfolder': '',
        'params': '1.3 B / 14 B',
        'weight_size': '3 GB (1.3B) / 30 GB (14B)',
        'sample_rate': 0,
        'languages': 'English (multilingual in 14B)',
        'license': 'Apache 2.0',
        'pip_deps': 'diffusers\ntransformers\naccelerate\nflash-attn --no-build-isolation\nimageio[ffmpeg]\nav',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**Text-to-Video** generation. The 1.3B variant fits on a 16 GB T4.',
        'quick_examples': 'A corgi running on a beach, cinematic, golden hour.\nA woman walking through Tokyo at night, neon lights.\nA timelapse of a flower blooming.',
    },
    'HunyuanVideo': {
        'name': 'HunyuanVideo',
        'title': 'HunyuanVideo \u2014 Tencent Text-to-Video',
        'subtitle': "Tencent's open text-to-video model. Apache 2.0. ~13 B parameters, A100-class GPU required.",
        'modality': 'video',
        'hf_repo': 'tencent/HunyuanVideo',
        'subfolder': '',
        'params': '13 B',
        'weight_size': '~30 GB',
        'sample_rate': 0,
        'languages': 'EN + ZH',
        'license': 'Apache 2.0',
        'pip_deps': 'torch\ndiffusers\ntransformers\naccelerate\nflash-attn --no-build-isolation\nimageio[ffmpeg]',
        'requires_gpu': True,
        'gated': False,
        'extra_features': '**High-quality** text-to-video. Best for short, cinematic clips.',
        'quick_examples': 'A red rose blooming in slow motion.\nA man walking through a snowy forest, dawn.\nA small bird flying over the ocean.',
    },
}
print(f'[Presets] {len(PRESETS)} curated model specs ready.')

In [ ]:

# @title STEP 4 — Generate Notebook
# @markdown Fill in the spec or load a preset, then click Generate. The output notebook is written
# @markdown to /content and a download link is shown.

import os, json, traceback
import gradio as gr
from pathlib import Path
from IPython.display import clear_output, display, FileLink

OUT_DIR = '/content'
os.makedirs(OUT_DIR, exist_ok=True)


def _load_preset(name):
    if name == '(none)':
        return [''] * 20
    spec = PRESETS.get(name, {})
    return [
        spec.get('name', ''),
        spec.get('title', ''),
        spec.get('subtitle', ''),
        spec.get('modality', 'tts'),
        spec.get('hf_repo', ''),
        spec.get('subfolder', ''),
        spec.get('params', ''),
        spec.get('weight_size', ''),
        spec.get('sample_rate', 24000),
        spec.get('languages', ''),
        spec.get('license', 'Apache 2.0'),
        spec.get('pip_deps', ''),
        spec.get('requires_gpu', True),
        spec.get('gated', False),
        spec.get('extra_features', ''),
        spec.get('quick_examples', ''),
        spec.get('hf_card_url', ''),
        spec.get('github_url', ''),
        spec.get('arxiv_url', ''),
        spec.get('citation', ''),
    ]


def _do_generate(name, title, subtitle, modality, hf_repo, subfolder, params, weight_size,
                 sample_rate, languages, license, pip_deps, requires_gpu, gated,
                 extra_features, quick_examples, hf_card_url, github_url, arxiv_url,
                 citation, progress=gr.Progress()):
    try:
        spec = {
            'name': (name or '').strip(),
            'title': (title or name or '').strip(),
            'subtitle': (subtitle or '').strip(),
            'modality': modality,
            'hf_repo': (hf_repo or '').strip(),
            'subfolder': (subfolder or '').strip(),
            'params': (params or '').strip(),
            'weight_size': (weight_size or '').strip(),
            'sample_rate': int(sample_rate) if modality == 'tts' else 0,
            'languages': (languages or '').strip(),
            'license': license,
            'pip_deps': (pip_deps or '').strip(),
            'requires_gpu': bool(requires_gpu),
            'gated': bool(gated),
            'extra_features': (extra_features or '').strip(),
            'quick_examples': (quick_examples or '').strip(),
            'hf_card_url': (hf_card_url or '').strip(),
            'github_url': (github_url or '').strip(),
            'arxiv_url': (arxiv_url or '').strip(),
            'citation': (citation or '').strip(),
        }
        if not spec['name']:
            return None, 'name is required', ''
        if not spec['hf_repo']:
            return None, 'hf_repo is required', ''
        if not spec['subtitle']:
            return None, 'subtitle is required', ''
        progress(0.2, desc='Building notebook ...')
        nb, errs = build_notebook(spec)
        if errs:
            return None, 'spec error: ' + '; '.join(errs), ''
        progress(0.6, desc='Writing file ...')
        out_path, fname = write_notebook(nb, OUT_DIR)
        size = os.path.getsize(out_path)
        with open(out_path) as f:
            nb2 = json.load(f)
        ids = [c['metadata'].get('id') for c in nb2['cells']]
        expected = ['view-in-github', 'header', 'step1-install', 'step2-cache', 'step3-core',
                    'step4-ui', 'step5-keepalive', 'step6-quicktest', 'step7-batch']
        ok_ids = ids == expected
        msg = 'wrote ' + out_path + ' (' + str(round(size/1024, 1)) + ' KB, ' + str(len(nb2['cells'])) + ' cells' + (', all IDs present' if ok_ids else ', MISSING IDS') + ')'
        return out_path, msg, out_path
    except Exception as e:
        traceback.print_exc()
        return None, 'FAILED: ' + str(e), ''


with gr.Blocks(title='Notebook Generator', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Notebook Generator\nScaffold a new 7-step AEI notebook from a spec.')
    with gr.Row():
        with gr.Column(scale=1):
            preset_dd = gr.Dropdown(choices=['(none)'] + sorted(PRESETS.keys()), value='(none)',
                                    label='Load preset',
                                    info='Pick a curated spec to auto-fill the form. You can edit any field after loading.')
        with gr.Column(scale=1):
            gr.Markdown('Fill in the spec \u2192')
    with gr.Row():
        with gr.Column():
            name_in = gr.Textbox(label='Name (slug)', placeholder='Trellis-3D',
                                 info='Used for the filename and the Python class name. No spaces.')
            title_in = gr.Textbox(label='Title (display)', placeholder="Trellis \u2014 Microsoft's Structured 3D Generation",
                                  info='H1 / README title. Can include em-dashes and punctuation.')
            subtitle_in = gr.Textbox(label='Subtitle (one-liner)', lines=2,
                                     placeholder='Structured 3D generation (Gaussian splats + meshes + NeRFs) ...',
                                     info='Markdown that goes right under the title. One or two sentences.')
            modality_in = gr.Dropdown(choices=MODALITIES, value='tts', label='Modality',
                                       info='Drives the Step 4 UI template (TTS -> audio, 3D -> mesh, video -> file, image -> file).')
            hf_repo_in = gr.Textbox(label='Hugging Face repo ID', placeholder='JeffreyXiang/TRELLIS-image-large',
                                    info='The `repo` argument to `snapshot_download`. Required.')
            subfolder_in = gr.Textbox(label='Subfolder (optional)', placeholder='',
                                      info='If weights are in a subfolder of the repo (e.g. `hunyuan3d-dit-v2-1`).')
        with gr.Column():
            params_in = gr.Textbox(label='Parameter count', placeholder='1.2 B',
                                   info='For the README and UI. e.g. `1.7B`, `99M`, `13 B`.')
            weight_in = gr.Textbox(label='Weight size', placeholder='3 GB',
                                   info='For the README and UI. e.g. `3 GB`, `~30 GB`.')
            sr_in = gr.Slider(minimum=0, maximum=48000, value=24000, step=1000, label='Sample rate (Hz)',
                              info='TTS only. 24000 = Qwen3/Higgs, 44100 = Supertonic/Dia, 48000 = dots.tts.')
            langs_in = gr.Textbox(label='Languages', placeholder='English (multilingual in 14B)',
                                  info='Free text for the README. `-` for 3D / video.')
            lic_in = gr.Dropdown(choices=LICENSES, value='Apache 2.0', label='License',
                                 info='Most open TTS / 3D / video models are Apache 2.0 or MIT.')
            gpu_in = gr.Checkbox(value=True, label='Requires GPU',
                                 info='Disable for CPU-only models (e.g. Supertonic-3 via ONNX).')
            gated_in = gr.Checkbox(value=False, label='Gated (needs HF_TOKEN)',
                                   info='Enable for Hugging Face-gated repos. Adds a Colab-secrets HF_TOKEN prologue.')
    pip_deps_in = gr.Textbox(label='pip install (one package per line)', lines=5,
                              placeholder='torch\ntransformers\nflash-attn --no-build-isolation',
                              info='Each line is one `pip install` package. Use `pkg @ git+https://...` for git installs.')
    extra_features_in = gr.Textbox(label='Extra features (optional, markdown)', lines=3,
                                    placeholder='**Voice cloning** from a 3+ second reference clip. ...',
                                    info='Goes into the header as bullet points. Plain markdown.')
    quick_examples_in = gr.Textbox(label='Quick-start examples (one per line)', lines=5,
                                     placeholder='Hello!\nThe quick brown fox.\nBonjour le monde.',
                                     info='Shown in the UI Examples gallery. Up to 5 lines.')
    with gr.Accordion('Optional: More info block (HF / GitHub / arXiv / citation)', open=False):
        hf_card_url_in = gr.Textbox(label='Hugging Face model card URL (optional)',
                                    placeholder='https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice',
                                    info='Default: https://huggingface.co/<repo_id>. Override if the model card is at a different URL.')
        github_url_in = gr.Textbox(label='GitHub repo URL (optional)',
                                   placeholder='https://github.com/QwenLM/Qwen3-TTS',
                                   info='The official source code repository. Used in the More info block.')
        arxiv_url_in = gr.Textbox(label='arXiv URL (optional)',
                                  placeholder='https://arxiv.org/abs/2501.12202',
                                  info='The research paper, if any. Used in the More info block.')
        citation_in = gr.Textbox(label='BibTeX citation (optional)', lines=5,
                                 placeholder='@misc{qwen3tts,\n  title={Qwen3-TTS},\n  ...\n}',
                                 info='Raw BibTeX. Shown inside a fenced code block in the More info section.')
    gen_btn = gr.Button('Generate notebook', variant='primary')
    with gr.Row():
        out_file = gr.File(label='Generated notebook (download here)')
        out_text = gr.Textbox(label='Status', lines=3, interactive=False)
    out_link = gr.Markdown('')

    preset_dd.change(
        _load_preset, inputs=[preset_dd],
        outputs=[name_in, title_in, subtitle_in, modality_in, hf_repo_in, subfolder_in,
                 params_in, weight_in, sr_in, langs_in, lic_in, pip_deps_in, gpu_in,
                 gated_in, extra_features_in, quick_examples_in,
                 hf_card_url_in, github_url_in, arxiv_url_in, citation_in],
    )
    gen_btn.click(
        _do_generate,
        inputs=[name_in, title_in, subtitle_in, modality_in, hf_repo_in, subfolder_in,
                params_in, weight_in, sr_in, langs_in, lic_in, pip_deps_in, gpu_in,
                gated_in, extra_features_in, quick_examples_in,
                hf_card_url_in, github_url_in, arxiv_url_in, citation_in],
        outputs=[out_file, out_text, out_link],
    ).then(
        lambda p: 'Open `/content/...` in the file browser, or [download here](' + str(p) + ').' if p else '',
        inputs=[out_file],
        outputs=[out_link],
    )
    demo.load(lambda: 'Pick a preset or fill in the spec, then click Generate.')

clear_output()
print('[UI] Launching generator ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.launch(share=True, show_error=True, height=900, prevent_thread_lock=False)
print('Generator ready. Use the form to scaffold a new model notebook.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/tencent/Hunyuan3D-2', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started.')

In [ ]:

# @title STEP 6 — Quick Test (validate the generator engine)
# @markdown Runs build_notebook() on the Qwen3-TTS preset and validates that the output is a well-formed
# @markdown 9-cell notebook.

import os, json, ast

spec = PRESETS['Qwen3-TTS-1.7B']
nb, errs = build_notebook(spec)
assert not errs, 'build errors: ' + str(errs)
assert len(nb['cells']) == 9, 'expected 9 cells, got ' + str(len(nb['cells']))

ids = [c['metadata'].get('id') for c in nb['cells']]
expected = ['view-in-github', 'header', 'step1-install', 'step2-cache', 'step3-core',
            'step4-ui', 'step5-keepalive', 'step6-quicktest', 'step7-batch']
for got, exp in zip(ids, expected):
    assert got == exp, 'cell id mismatch: ' + str(got) + ' != ' + str(exp)


def _clean(src):
    out = []
    lines = src.split(chr(10))
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.lstrip().startswith(('# @title', '# @markdown', '!')):
            i += 1
            continue
        out.append(line)
        i += 1
    return chr(10).join(out)


for i, c in enumerate(nb['cells']):
    if c['cell_type'] == 'code':
        try:
            ast.parse(_clean(''.join(c['source'])))
        except SyntaxError as e:
            print('cell ' + str(i) + ' SYNTAX at line ' + str(e.lineno) + ': ' + e.msg)
            raise

out_path, fname = write_notebook(nb, '/tmp')
size = os.path.getsize(out_path)
print('Qwen3-TTS-1.7B spec generated OK')
print('  file: ' + out_path + '  (' + str(round(size/1024, 1)) + ' KB)')
print('  cells: ' + str(len(nb['cells'])))
print('  IDs: ' + str(ids))
print('Generator engine is verified working.')

In [ ]:

# @title STEP 7 — Batch-generate all presets
# @markdown One click: emits a notebook for every preset in Step 3.

import os, json, traceback
out_dir = '/content'
ok = 0
fail = 0
failed = []
for name, spec in PRESETS.items():
    try:
        nb, errs = build_notebook(spec)
        if errs:
            print('[' + name + '] spec: ' + str(errs))
            fail += 1
            failed.append(name)
            continue
        out_path, fname = write_notebook(nb, out_dir)
        size = os.path.getsize(out_path)
        print('[' + name + '] ' + fname + ' (' + str(round(size/1024, 1)) + ' KB)')
        ok += 1
    except Exception as e:
        print('[' + name + '] EXCEPTION: ' + str(e))
        traceback.print_exc()
        fail += 1
        failed.append(name)
print('DONE: ' + str(ok) + ' OK / ' + str(fail) + ' failed / ' + str(len(PRESETS)) + ' total')
if failed:
    print('Failed:', failed)